# M5 · Cache simulator with Random, LRU and LFU

Each layer gets a cache of **K slots** (K out of 64 experts fit on the GPU).
For every token and every layer:
- the 8 needed experts are already cached → **hit** (free)
- not cached → **miss**: copy it from CPU (slow), evicting another expert if the cache is full

The **policy** chooses which expert to evict:
- **Random**: any expert (baseline)
- **LRU**: the one *least recently used*
- **LFU**: the one *least frequently used*
- **NoCache**: flush everything after each token (every request is a copy), our "no caching" baseline

The code is in `src/simulator.py` (`LayerCache`, `simulate`) and `src/policies.py`.

**Kaggle:** Accelerator = **None** (CPU is enough, saves GPU quota), Internet = On.
**Inputs:** Add Input → Datasets → **moe-traces** (from M2).

Optional: also add the **M3 notebook output** so the measured copy time is used.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

## 1. Watch LRU work on a tiny example
One layer, 2 slots, one expert per token. Request sequence: 1, 2, 1, 3, 1, 2.

In [ ]:
from src.simulator import LayerCache
from src.policies import LRUPolicy

cache = LayerCache(slots=2, policy=LRUPolicy())
for t, expert in enumerate([1, 2, 1, 3, 1, 2]):
    loads = cache.request([expert], t)           # list of (loaded expert, evicted expert)
    result = "HIT " if not loads else f"MISS (load {expert}, evict {loads[0][1]})"
    print(f"token {t}: needs {expert} -> {result:24s} cache now {sorted(cache.cache)}")
print("hits:", cache.hits, "misses:", cache.misses)

Check by hand: at token 3, expert 2 was used longer ago than expert 1, so LRU evicts 2.

## 2. Run the policies on the real traces
We use the **second half** of each trace (the first half is kept for "learning" in M6/M7).
Set `TEST_TOKENS = 20_000` for a quick try; `None` uses the whole test half (~10 min).

In [ ]:
import os
import pandas as pd
import config
from src.files import load_trace
from src.simulator import simulate, split_trace
from src.calibrate import load_copy_time

TEST_TOKENS = None          # e.g. 20_000 for a quick run
copy_time = load_copy_time()
print(f"copy time per expert: {copy_time*1000:.2f} ms")

rows = []
for dataset in ["wikitext", "code"]:
    fit, test = split_trace(load_trace(dataset))
    test = test[:TEST_TOKENS]
    for slots in config.CACHE_SIZES:
        for policy in ["NoCache", "Random", "LRU", "LFU"]:
            r = simulate(test, policy, slots, copy_time_s=copy_time)
            rows.append({"dataset": dataset, "slots": slots, "policy": policy,
                         "hit_rate": r["hit_rate"], "misses": r["misses"],
                         "stall_ms_per_token": r["stall_ms_per_token"]})
        print(dataset, "slots", slots, "done")

basic = pd.DataFrame(rows)
basic.to_csv(os.path.join(config.RESULTS_DIR, "M5_basic_policies.csv"), index=False)
basic.pivot_table(index=["dataset", "slots"], columns="policy", values="hit_rate").round(3)

## 3. Hit rate vs cache size

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, dataset in zip(axes, ["wikitext", "code"]):
    for policy, g in basic[basic.dataset == dataset].groupby("policy"):
        ax.plot(100 * g.slots / 64, g.hit_rate, marker="o", label=policy)
    ax.set(title=dataset, xlabel="cache size (% of the 64 experts per layer)", ylabel="hit rate")
    ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(config.RESULTS_DIR, "M5_hit_rate.png"), dpi=130); plt.show()

**Note:** with 8 slots every policy gives the same result. Each token needs 8 experts,
so the cache can only ever hold the current token's experts and there is nothing to choose.

## ✅ Done when
- `!python -m pytest -q tests` passes (it checks the simulator on hand-made traces), and
- the table above is filled for 3 cache sizes × 2 datasets.